In [3]:
!pip install tensorflow


  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.1.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 MB 12.2 MB/s  0:00:16m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 17.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.4/667.4 kB 13.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 7.6 MB/s  0:00:00 eta 0:00:01m
Using cached tensorboard_data_server-0.7.2-py3-none-any.whl (2.4 kB)
Using cached astunparse-1.6.3-py2.py3-none-any.whl (12 kB)
Using cached gast-0.6.0-py

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
from tensorflow.keras.optimizers import AdamW

df = pd.read_csv("tweet_full.csv")  # your uploaded file

print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))

topic_cols = [col for col in df.columns if col.startswith("topic_")]
sent_cols = [c for c in ["min_sentiment", "max_sentiment", "avg_sentiment"] if c in df.columns]

base_cols = []
for c in ["volume", "T_minus_1_price", "T_price", "close", "adjclose"]:
    if c in df.columns:
        base_cols.append(c)

features = base_cols + topic_cols + sent_cols
features = list(dict.fromkeys(features)) 

target_col = None
for cand in ["change_vs_T_minus_1_price", "change_vs_T_minus_1"]:
    if cand in df.columns:
        target_col = cand
        break

if target_col is None and "T_price" in df.columns and "T_minus_1_price" in df.columns:
    df["change_vs_T_minus_1_price"] = df["T_price"] - df["T_minus_1_price"]
    target_col = "change_vs_T_minus_1_price"

if target_col is None:
    raise ValueError("Could not find or derive a suitable target column.")

print(f"Using target: {target_col}")
print(f"Using {len(features)} features: {features}")

X = df[features].fillna(0)
y = df[target_col].fillna(0)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, shuffle=False)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

initial_lr = 1e-3
warmup_ratio = 0.1
total_epochs = 100 

def lr_schedule(epoch):
    if epoch < warmup_ratio * total_epochs:
        return initial_lr * (epoch + 1) / (warmup_ratio * total_epochs)
    else:
        return initial_lr * np.exp(-0.2 * (epoch - warmup_ratio * total_epochs))

lr_scheduler = LearningRateScheduler(lr_schedule)

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='linear') 
])

optimizer = AdamW(learning_rate=initial_lr, weight_decay=1e-4)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=total_epochs,
    batch_size=16,
    callbacks=[lr_scheduler], 
    verbose=1
)

loss, mae = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\n✅ Test MSE: {loss:.6f}, Test MAE: {mae:.6f}")

y_pred = model.predict(X_test_scaled).ravel()
results = pd.DataFrame({
    "y_true": y_test.values,
    "y_pred": y_pred
})
results["error"] = results["y_true"] - results["y_pred"]

print("\nSample predictions:")
print(results.head())



Dataset shape: (2141, 19)
Columns: ['close', 'adjclose', 'volume', 'datetime', 'ticker', 'T_minus_1_price', 'T_price', 'change_vs_T_minus_1', 'topic_0_score', 'topic_1_score', 'topic_2_score', 'topic_3_score', 'topic_4_score', 'topic_5_score', 'topic_6_score', 'topic_7_score', 'min_sentiment', 'max_sentiment', 'avg_sentiment']
Using target: change_vs_T_minus_1
Using 16 features: ['volume', 'T_minus_1_price', 'T_price', 'close', 'adjclose', 'topic_0_score', 'topic_1_score', 'topic_2_score', 'topic_3_score', 'topic_4_score', 'topic_5_score', 'topic_6_score', 'topic_7_score', 'min_sentiment', 'max_sentiment', 'avg_sentiment']
Epoch 1/100


/opt/anaconda3/lib/python3.11/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


107/107 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.5851 - mae: 0.6037 - val_loss: 0.3507 - val_mae: 0.4746 - learning_rate: 1.0000e-04
Epoch 2/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3275 - mae: 0.5151 - val_loss: 0.2621 - val_mae: 0.4876 - learning_rate: 2.0000e-04
Epoch 3/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 959us/step - loss: 0.2802 - mae: 0.5011 - val_loss: 0.2513 - val_mae: 0.4890 - learning_rate: 3.0000e-04
Epoch 4/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 977us/step - loss: 0.2709 - mae: 0.4957 - val_loss: 0.2555 - val_mae: 0.4865 - learning_rate: 4.0000e-04
Epoch 5/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 962us/step - loss: 0.2664 - mae: 0.4962 - val_loss: 0.2507 - val_mae: 0.4919 - learning_rate: 5.0000e-04
Epoch 6/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 980us/step - loss: 0.2642 - mae: 0.4959 - val_loss: 0.2550 - val_mae: 0.4873 - learning_rate: 6.0000e-04
Epoch 7/100
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 967us/step - loss: 0.2608 - mae: 0.4936 - val_loss: 0.2546 - val_mae: 0.4887 -

In [7]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_binary = (results['y_pred'] > 0.5).astype(int)
y_true_binary = results['y_true'].astype(int)

cm = confusion_matrix(y_true_binary, y_pred_binary)
print("Confusion Matrix [[TN, FP], [FN, TP]]:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_true_binary, y_pred_binary, digits=4))


Confusion Matrix [[TN, FP], [FN, TP]]:
[[55 53]
 [45 62]]

Classification Report:
              precision    recall  f1-score   support

           0     0.5500    0.5093    0.5288       108
           1     0.5391    0.5794    0.5586       107

    accuracy                         0.5442       215
   macro avg     0.5446    0.5443    0.5437       215
weighted avg     0.5446    0.5442    0.5436       215

